# 02 - Rating Systems

This notebook computes ELO and Glicko-2 ratings from historical bout data.

**Rating Systems:**
- **ELO**: Standard chess-like rating with configurable K-factor
- **Glicko-2**: Adds rating deviation (uncertainty) and volatility measures

**Key Principle:** Ratings are computed using ONLY information available BEFORE each bout.

**Inputs:**
- `matches.parquet` from notebook 01

**Outputs:**
- `matches_with_ratings.parquet` - Matches with pre-bout ratings

In [ ]:
# Environment setup
import sys
import os

INPUT_PATH = '/kaggle/input/sumo-data-01' if os.path.exists('/kaggle/input') else './output'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
matches_df = pd.read_parquet(f"{INPUT_PATH}/matches.parquet")
print(f"Loaded {len(matches_df):,} matches")
print(f"Columns: {matches_df.columns.tolist()}")
display(matches_df.head())

In [ ]:
# Check data range
if 'bashoId' in matches_df.columns:
    print(f"Basho range: {matches_df['bashoId'].min()} to {matches_df['bashoId'].max()}")
    
# Count unique wrestlers
east_ids = set(matches_df['eastId'].unique())
west_ids = set(matches_df['westId'].unique())
all_ids = east_ids | west_ids
print(f"Unique wrestlers: {len(all_ids):,}")

## ELO Rating System

Standard ELO implementation:
- Initial rating: 1500
- K-factor: 32 (how much ratings change per game)
- Expected score formula: E = 1 / (1 + 10^((Rb-Ra)/400))

In [ ]:
@dataclass
class EloRating:
    rating: float = 1500.0
    games_played: int = 0


class EloSystem:
    def __init__(self, k_factor: float = 32.0, initial_rating: float = 1500.0):
        self.k_factor = k_factor
        self.initial_rating = initial_rating
        self.ratings: Dict[int, EloRating] = defaultdict(
            lambda: EloRating(rating=self.initial_rating)
        )

    def expected_score(self, rating_a: float, rating_b: float) -> float:
        return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))

    def update(self, winner_id: int, loser_id: int) -> Tuple[float, float]:
        winner = self.ratings[winner_id]
        loser = self.ratings[loser_id]

        expected_winner = self.expected_score(winner.rating, loser.rating)
        expected_loser = 1.0 - expected_winner

        winner.rating += self.k_factor * (1.0 - expected_winner)
        loser.rating += self.k_factor * (0.0 - expected_loser)

        winner.games_played += 1
        loser.games_played += 1

        return winner.rating, loser.rating

    def get_rating(self, wrestler_id: int) -> float:
        return self.ratings[wrestler_id].rating

## Glicko-2 Rating System

Glicko-2 adds:
- **Rating Deviation (RD)**: Uncertainty in the rating. Higher = less certain.
- **Volatility**: How erratic the player's performance is.

Key behaviors:
- RD increases during periods of inactivity
- RD decreases as more games are played
- Volatility captures whether a player performs consistently

In [ ]:
@dataclass
class Glicko2Rating:
    mu: float = 0.0  # Internal scale
    phi: float = 350.0 / 173.7178  # Initial RD
    sigma: float = 0.06  # Initial volatility

    @property
    def rating(self) -> float:
        return self.mu * 173.7178 + 1500.0

    @property
    def rd(self) -> float:
        return self.phi * 173.7178

    @classmethod
    def from_standard(cls, rating: float = 1500.0, rd: float = 350.0,
                     sigma: float = 0.06) -> 'Glicko2Rating':
        return cls(
            mu=(rating - 1500.0) / 173.7178,
            phi=rd / 173.7178,
            sigma=sigma
        )


class Glicko2System:
    def __init__(self, tau: float = 0.5, initial_rating: float = 1500.0,
                 initial_rd: float = 350.0, initial_volatility: float = 0.06):
        self.tau = tau
        self.initial_rating = initial_rating
        self.initial_rd = initial_rd
        self.initial_volatility = initial_volatility

        self.ratings: Dict[int, Glicko2Rating] = defaultdict(
            lambda: Glicko2Rating.from_standard(
                self.initial_rating, self.initial_rd, self.initial_volatility
            )
        )

    def _g(self, phi: float) -> float:
        return 1.0 / math.sqrt(1.0 + 3.0 * phi ** 2 / math.pi ** 2)

    def _E(self, mu: float, mu_j: float, phi_j: float) -> float:
        return 1.0 / (1.0 + math.exp(-self._g(phi_j) * (mu - mu_j)))

    def update_single_bout(self, winner_id: int, loser_id: int) -> None:
        """Simplified single-bout update."""
        winner = self.ratings[winner_id]
        loser = self.ratings[loser_id]

        g_w = self._g(winner.phi)
        g_l = self._g(loser.phi)

        E_w = self._E(winner.mu, loser.mu, loser.phi)
        E_l = self._E(loser.mu, winner.mu, winner.phi)

        # Update ratings
        winner.mu += 0.5 * g_l * (1.0 - E_w)
        loser.mu += 0.5 * g_w * (0.0 - E_l)

        # Reduce RD after bout
        winner.phi = max(winner.phi * 0.98, 30.0 / 173.7178)
        loser.phi = max(loser.phi * 0.98, 30.0 / 173.7178)

    def get_rating(self, wrestler_id: int) -> Tuple[float, float, float]:
        r = self.ratings[wrestler_id]
        return r.rating, r.rd, r.sigma

## Compute Ratings Chronologically

Process all bouts in chronological order, storing the rating BEFORE each bout.

In [ ]:
def compute_all_ratings(df: pd.DataFrame, 
                        elo_k: float = 32.0,
                        glicko_tau: float = 0.5) -> pd.DataFrame:
    """
    Compute ELO and Glicko-2 ratings for all bouts.
    
    CRITICAL: We store ratings BEFORE the bout occurs to prevent data leakage.
    """
    # Sort chronologically
    df = df.copy()
    df = df.sort_values(['bashoId', 'day']).reset_index(drop=True)
    
    # Initialize systems
    elo = EloSystem(k_factor=elo_k)
    glicko = Glicko2System(tau=glicko_tau)
    
    # Result columns
    n = len(df)
    east_elo = np.zeros(n)
    west_elo = np.zeros(n)
    east_glicko_rating = np.zeros(n)
    east_glicko_rd = np.zeros(n)
    east_glicko_vol = np.zeros(n)
    west_glicko_rating = np.zeros(n)
    west_glicko_rd = np.zeros(n)
    west_glicko_vol = np.zeros(n)
    
    for idx, row in df.iterrows():
        if idx % 50000 == 0:
            print(f"Processing bout {idx:,} / {n:,}")
        
        east_id = row['eastId']
        west_id = row['westId']
        winner_id = row.get('winnerId')
        
        # Store ratings BEFORE bout
        east_elo[idx] = elo.get_rating(east_id)
        west_elo[idx] = elo.get_rating(west_id)
        
        g_e = glicko.get_rating(east_id)
        g_w = glicko.get_rating(west_id)
        
        east_glicko_rating[idx] = g_e[0]
        east_glicko_rd[idx] = g_e[1]
        east_glicko_vol[idx] = g_e[2]
        west_glicko_rating[idx] = g_w[0]
        west_glicko_rd[idx] = g_w[1]
        west_glicko_vol[idx] = g_w[2]
        
        # Update ratings (if valid result)
        if winner_id and (winner_id == east_id or winner_id == west_id):
            if winner_id == east_id:
                elo.update(east_id, west_id)
                glicko.update_single_bout(east_id, west_id)
            else:
                elo.update(west_id, east_id)
                glicko.update_single_bout(west_id, east_id)
    
    # Add columns
    df['east_elo'] = east_elo
    df['west_elo'] = west_elo
    df['east_glicko_rating'] = east_glicko_rating
    df['east_glicko_rd'] = east_glicko_rd
    df['east_glicko_vol'] = east_glicko_vol
    df['west_glicko_rating'] = west_glicko_rating
    df['west_glicko_rd'] = west_glicko_rd
    df['west_glicko_vol'] = west_glicko_vol
    
    # Compute differentials
    df['elo_diff'] = df['east_elo'] - df['west_elo']
    df['glicko_rating_diff'] = df['east_glicko_rating'] - df['west_glicko_rating']
    df['glicko_rd_diff'] = df['east_glicko_rd'] - df['west_glicko_rd']
    
    print(f"Done! Processed {n:,} bouts.")
    return df

In [ ]:
# Compute ratings
matches_with_ratings = compute_all_ratings(
    matches_df, 
    elo_k=32.0,
    glicko_tau=0.5
)

## Validate Ratings

In [ ]:
# Check rating distributions
print("=== ELO Distribution ===")
print(f"East ELO - Mean: {matches_with_ratings['east_elo'].mean():.1f}, "
      f"Std: {matches_with_ratings['east_elo'].std():.1f}")
print(f"West ELO - Mean: {matches_with_ratings['west_elo'].mean():.1f}, "
      f"Std: {matches_with_ratings['west_elo'].std():.1f}")

print("\n=== Glicko-2 Distribution ===")
print(f"East Rating - Mean: {matches_with_ratings['east_glicko_rating'].mean():.1f}, "
      f"Std: {matches_with_ratings['east_glicko_rating'].std():.1f}")
print(f"East RD - Mean: {matches_with_ratings['east_glicko_rd'].mean():.1f}, "
      f"Std: {matches_with_ratings['east_glicko_rd'].std():.1f}")

In [ ]:
# Check predictive power of ratings
df_valid = matches_with_ratings[
    matches_with_ratings['winnerId'].notna() & 
    (matches_with_ratings['east_elo'] != 1500) &
    (matches_with_ratings['west_elo'] != 1500)
].copy()

df_valid['east_won'] = (df_valid['winnerId'] == df_valid['eastId']).astype(int)
df_valid['elo_predicts_east'] = (df_valid['elo_diff'] > 0).astype(int)
df_valid['glicko_predicts_east'] = (df_valid['glicko_rating_diff'] > 0).astype(int)

elo_accuracy = (df_valid['elo_predicts_east'] == df_valid['east_won']).mean()
glicko_accuracy = (df_valid['glicko_predicts_east'] == df_valid['east_won']).mean()

print(f"\n=== Rating Prediction Accuracy ===")
print(f"ELO (higher rating wins): {elo_accuracy:.1%}")
print(f"Glicko-2 (higher rating wins): {glicko_accuracy:.1%}")
print(f"\n(Note: 50% = random, >60% = useful signal)")

In [ ]:
# Sample of final data
print("\nSample with ratings:")
display(matches_with_ratings[
    ['bashoId', 'day', 'eastId', 'westId', 'winnerId',
     'east_elo', 'west_elo', 'elo_diff',
     'east_glicko_rating', 'east_glicko_rd',
     'west_glicko_rating', 'west_glicko_rd']
].tail(10))

## Compute Average Rating by Banzuke Rank

This will be used to calculate "rating minus expected for rank" features.

In [ ]:
# Stack east and west data
east_data = matches_with_ratings[['eastId', 'east_rank_numeric', 'east_elo', 'east_glicko_rating']].copy()
east_data.columns = ['wrestler_id', 'rank_numeric', 'elo', 'glicko_rating']

west_data = matches_with_ratings[['westId', 'west_rank_numeric', 'west_elo', 'west_glicko_rating']].copy()
west_data.columns = ['wrestler_id', 'rank_numeric', 'elo', 'glicko_rating']

all_data = pd.concat([east_data, west_data], ignore_index=True)

# Filter to valid ranks and non-initial ratings
all_data = all_data[
    (all_data['rank_numeric'] < 200) &
    (all_data['elo'] != 1500)
]

# Compute average by rank
rank_averages = all_data.groupby('rank_numeric').agg({
    'elo': 'mean',
    'glicko_rating': 'mean'
}).reset_index()
rank_averages.columns = ['rank_numeric', 'avg_elo_for_rank', 'avg_glicko_for_rank']

print("Average ratings by rank (sample):")
display(rank_averages.head(20))

In [ ]:
# Save rank averages for use in feature engineering
rank_averages.to_parquet(f"{OUTPUT_PATH}/rank_averages.parquet", index=False)
print(f"Saved rank averages to {OUTPUT_PATH}/rank_averages.parquet")

## Save Data

In [ ]:
matches_with_ratings.to_parquet(f"{OUTPUT_PATH}/matches_with_ratings.parquet", index=False)
print(f"Saved {len(matches_with_ratings):,} matches with ratings to {OUTPUT_PATH}/matches_with_ratings.parquet")

In [ ]:
# Verify
print(f"\nFinal columns: {matches_with_ratings.columns.tolist()}")
print(f"\nNew rating columns:")
rating_cols = [c for c in matches_with_ratings.columns if 'elo' in c.lower() or 'glicko' in c.lower()]
print(rating_cols)

## Next Steps

The data with ratings is ready for:
- `03_feature_engineering.ipynb` - Build the full feature set